# 🍷 Atividade Avaliativa — Previsão de Qualidade de Vinho Tinto

 **Tempo estimado:** até 2 horas | **Total:** 10 pontos

---

## Instruções

Este notebook contém um pipeline de Ciência de Dados com **erros propositais** e **lacunas** para completar.

| Marcador | O que fazer | Qtd | Pontos |
|----------|-------------|:---:|-------:|
| 🔴 `# BUG:` | Encontre e **corrija** o erro na linha indicada | 3 | 1,0 pt cada |
| 🟡 `# TODO:` | **Preencha** o `___` com o código correto | 5 | 0,5 pt cada |
| 🔵 **Questão** | **Responda** na célula Markdown indicada | 3 | 1,5 pt cada |


---

## Contexto do Problema

**Dataset:** [Wine Quality Red](https://archive.ics.uci.edu/ml/datasets/wine+quality) — UCI Machine Learning Repository  
**Objetivo:** classificar vinhos tintos como *boa qualidade* (nota ≥ 6) ou *baixa qualidade* (nota < 6).

| Feature | Descrição |
|---------|-----------|
| `fixed acidity` | Acidez fixa (g/L) |
| `volatile acidity` | Acidez volátil (g/L) |
| `citric acid` | Ácido cítrico (g/L) |
| `residual sugar` | Açúcar residual (g/L) |
| `chlorides` | Cloretos (g/L) |
| `free sulfur dioxide` | SO₂ livre (mg/L) |
| `total sulfur dioxide` | SO₂ total (mg/L) |
| `density` | Densidade (g/cm³) |
| `pH` | pH |
| `sulphates` | Sulfatos (g/L) |
| `alcohol` | Teor alcoólico (% vol.) |
| `quality` | Nota dos especialistas — **será transformada no target** |


In [ ]:
# ─── Importações — não altere esta célula ────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
print("✅ Importações concluídas!")

---
## 1. Carregamento dos Dados

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

# TODO [T1]: o arquivo CSV usa ponto-e-vírgula (;) como separador de colunas.
# Adicione o argumento correto para o parâmetro `sep`:
df = pd.read_csv(url, ___)

print(f"Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")
df.head()

In [ ]:
print("=== Tipos de dados ===")
print(df.dtypes)
print()
print("=== Estatísticas Descritivas ===")
df.describe().round(2)

---
## 2. Limpeza dos Dados

In [ ]:
# Valores ausentes e duplicatas
print("Valores ausentes por coluna:")
print(df.isnull().sum())
print()

before = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Duplicatas removidas: {before - df.shape[0]}  →  {df.shape[0]} linhas restantes")

In [ ]:
# Detecção e remoção de outliers na feature 'total sulfur dioxide' via método IQR
col = "total sulfur dioxide"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

n_outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()

print(f"Q1 = {Q1:.1f}  |  Q3 = {Q3:.1f}  |  IQR = {IQR:.1f}")
print(f"Limite inferior: {lim_inf:.1f}  |  Limite superior: {lim_sup:.1f}")
print(f"Outliers detectados em '{col}': {n_outliers}")
print()

# BUG [B1]: a condição abaixo deveria manter apenas amostras dentro dos limites normais
# (ou seja, sem outliers).
# Identifique o erro, adicione um comentário explicando-o e corrija.
df = df[(df[col] >= lim_sup) | (df[col] <= lim_inf)].reset_index(drop=True)

print(f"Amostras restantes após remoção: {df.shape[0]}")


---
## 3. Análise Exploratória de Dados (EDA)

### 3.1 Distribuição da variável alvo original

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df["quality"].value_counts().sort_index().plot(
    kind="bar", ax=ax, color="steelblue", edgecolor="black"
)
ax.set_title("Distribuição das Notas de Qualidade")
ax.set_xlabel("Nota (0–10)")
ax.set_ylabel("Contagem")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

### 3.2 Criação da variável alvo binária

In [ ]:
# TODO [T2]: crie a coluna 'target' como variável binária.
# Vinhos com qualidade >= 6 recebem 1 (boa qualidade), demais recebem 0.
df["target"] = (df["quality"] ___ 6).astype(int)

print("Distribuição do target binário:")
print(df["target"].value_counts())
print(f"\nProporção de boa qualidade: {df['target'].mean():.1%}")

### 3.3 Correlação entre features e target

In [ ]:
features = df.drop(columns=["quality", "target"])
corr_matrix = features.assign(target=df["target"]).corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, square=True)
plt.title("Mapa de Correlação — Features e Target")
plt.tight_layout()
plt.show()

### 3.4 Teor alcoólico por classe de qualidade

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="target", y="alcohol", palette="Set2")
plt.xticks([0, 1], ["Baixa Qualidade (0)", "Boa Qualidade (1)"])
plt.title("Teor Alcoólico por Classe de Qualidade")
plt.xlabel("Classe")
plt.ylabel("Álcool (% vol.)")
plt.tight_layout()
plt.show()

### 🔵 Questão 1 — Análise Exploratória

**Com base no mapa de correlação e no boxplot, responda:**

 Qual feature apresenta a **maior correlação positiva** com o `target`?  
   O que isso indica para a tarefa de classificação? Há alguma feature com correlação negativa relevante?

> ✏️ *Escreva sua resposta aqui:*


---
## 4. Pré-processamento

In [ ]:
X = df.drop(columns=["quality", "target"])
y = df["target"]

print(f"Shape de X: {X.shape}  |  Shape de y: {y.shape}")
print(f"\nFeatures ({X.shape[1]}): {list(X.columns)}")

In [ ]:
# BUG [B2]: o código abaixo contém um problema conceitual no fluxo de pré-processamento.
# Identifique o problema, adicione um comentário explicando-o e corrija.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# TODO [T3]: complete o argumento que garante que a divisão treino/teste preserve
# a proporção das classes em ambos os conjuntos. Preencha ___ com o argumento correto.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, ___
)

X_train_scaled = X_train
X_test_scaled  = X_test

print(f"Treino: {X_train_scaled.shape[0]} amostras | Teste: {X_test_scaled.shape[0]} amostras")
print(f"\nProporção de boa qualidade — Treino: {y_train.mean():.1%} | Teste: {y_test.mean():.1%}")

---
## 5. Treinamento do Modelo

In [ ]:
model = LogisticRegression(max_iter=1000)

# TODO [T4]: chame o método correto para treinar o modelo.
# Preencha o nome do método no lugar de ___
model.___(X_train_scaled, y_train)

print("✅ Modelo treinado!")

---
## 6. Avaliação do Modelo

In [ ]:
# TODO [T5]: gere as previsões do modelo para o conjunto de teste.
y_pred = model.___(X_test_scaled)

print(f"Previsões geradas: {len(y_pred)} amostras")

In [ ]:
print("=== Relatório de Classificação ===")
print(classification_report(
    y_test, y_pred,
    target_names=["Baixa Qualidade", "Boa Qualidade"],
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Baixa Qualidade", "Boa Qualidade"],
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de Confusão")
plt.tight_layout()
plt.show()

f1 = f1_score(y_test, y_pred, average="macro")
print(f"F1-Score (macro): {f1:.4f}")

### 6.2 Curva ROC e AUC

In [ ]:
# BUG [B3] ⭐ O código abaixo calcula e exibe o ROC-AUC,
# mas há um problema conceitual que torna o valor obtido incorreto
# como medida real de discriminação do modelo.
# O código roda sem erros — analise cuidadosamente o que cada linha calcula.
# Identifique o problema, explique em um comentário detalhado e corrija.

y_scores = model.predict(X_test_scaled)

roc_auc = roc_auc_score(y_test, y_scores)
print(f"ROC-AUC: {roc_auc:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, y_scores, ax=ax, name="Regressão Logística")
ax.set_title("Curva ROC")
plt.tight_layout()
plt.show()

### 🔵 Questão 2 — Interpretação das Métricas

**Com base nos resultados acima, responda:**

a. O dataset é **balanceado**? Como o argumento que você adicionou em T3 mitiga um possível problema na avaliação?

b. Por que a **acurácia** pode ser enganosa neste problema? Qual(is) métrica(s) do relatório de classificação são mais adequadas e por quê?

c. Qual é a diferença conceitual entre `model.predict()` e `model.predict_proba()[:, 1]`?  
   Por que essa diferença é crítica ao calcular o **ROC-AUC**? O que acontece com a curva ROC quando se usa o método incorreto?

> ✏️ *Escreva sua resposta aqui:*


---
## 7. Reflexão Final

### 🔵 Questão 3 — Proposta de Melhoria

**Com base em todo o pipeline realizado, identifique ao menos duas limitações ou pontos de melhoria  
e proponha soluções concretas e justificadas.**

Considere aspectos como:
- Qualidade e pré-processamento dos dados (ex: tratamento de outliers, engenharia de features)
- Estratégias para lidar com desbalanceamento de classes
- Técnicas de validação mais robustas do que o simples hold-out
- Alternativas de modelagem


> ✏️ *Escreva sua resposta aqui:*


---
*Salve o notebook com suas respostas e envie conforme orientação do professor.*